# 🦙 Ollama over NetBird Quickstart Guide

This notebook demonstrates how to connect to the **AIT Brainlab Ollama Service** on server `la` across the **NetBird WireGuard Mesh** (`http://la:11434`).

### Prerequisites:
- Connected to NetBird VPN (`https://netbird.brain.cs.ait.ac.th`)
- `pip install ollama pandas`

> **Note for JupyterLab on Server `la`**: If running inside a container on server `la` itself, you can also use `http://ollama:11434`.

In [22]:
# Install prerequisites
%pip install -q ollama pandas

/mnt/pool-1/home/akraradets/projects/AIT-brainlab/brainlab-base/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
from ollama import Client
import pandas as pd

# Initialize client targeting server 'la' over NetBird MagicDNS
# (If inside a JupyterHub container on 'la', use 'http://ollama:11434')
OLLAMA_HOST = "http://la:11434"
client = Client(host=OLLAMA_HOST)

print(f"Connected to Ollama at {OLLAMA_HOST}")

Connected to Ollama at http://la:11434


## 1. List Available Models
View all models currently downloaded and available on the server (`client.list()`).

In [2]:
models_resp = client.list()

model_rows = []
for m in models_resp.get("models", []):
    size = (m.get("size") if isinstance(m, dict) else getattr(m, "size", 0)) or 0
    size_gb = size / (1024 ** 3)
    details = (m.get("details") if isinstance(m, dict) else getattr(m, "details", {})) or {}
    modified = m.get("modified_at") if isinstance(m, dict) else getattr(m, "modified_at", None)
    
    model_name = (m.get("model") or m.get("name")) if isinstance(m, dict) else (getattr(m, "model", None) or getattr(m, "name", "N/A"))
    family = details.get("family") if isinstance(details, dict) else getattr(details, "family", "N/A")
    param_size = details.get("parameter_size") if isinstance(details, dict) else getattr(details, "parameter_size", "N/A")
    quant = details.get("quantization_level") if isinstance(details, dict) else getattr(details, "quantization_level", "N/A")
    
    model_rows.append({
        "Model": model_name,
        "Size": f"{size_gb:.2f} GB",
        "Family": family or "N/A",
        "Parameter Size": param_size or "N/A",
        "Quantization": quant or "N/A",
        "Modified": str(modified)[:19] if modified else "N/A",
    })

df_models = pd.DataFrame(model_rows)
display(df_models)

,Model,Size,Family,Parameter Size,Quantization,Modified
0,qwen3.8:27b,16.52 GB,qwen35,27.3B,Q4_K_M,2026-11-16 02:34:49
1,granite3.3:8b,4.60 GB,granite,8.2B,Q4_K_M,2026-11-11 22:30:41
2,olmo2:13b,7.78 GB,olmo2,13.7B,Q4_K_M,2026-11-11 22:27:48
3,mistral-small3.2:24b,14.14 GB,mistral3,24.0B,Q4_K_M,2026-11-11 22:23:04
4,gemma4:12b,7.47 GB,gemma4,11.9B,Q4_K_M,2026-11-11 07:23:28
5,gpt-oss:20b,12.85 GB,gpt-oss,20.9B,MXFP4,2026-11-11 03:10:16
6,gemma3:12b,7.60 GB,gemma3,11.8B,Q4_K_M,2026-11-11 03:05:26
7,phi4:14b,8.43 GB,phi3,14.7B,Q4_K_M,2026-11-11 03:03:37
8,llama3.2:1b,1.23 GB,llama,1.2B,Q8_0,2026-11-10 18:27:28


## 2. See Active (Loaded) Models in GPU VRAM
Check which models are currently consuming GPU memory (`client.ps()`).

In [3]:
# Ollama is allocated 44 GB on GPU 1 (RTX A6000 48GB with 4GB safety reserve)
OLLAMA_VRAM_BUDGET_GB = 44.0

ps_resp = client.ps()
models = ps_resp.get("models", [])

# Calculate VRAM consumption & available headroom across loaded models
total_vram_used = sum((m.get("size_vram") if isinstance(m, dict) else getattr(m, "size_vram", 0)) or 0 for m in models)
total_vram_used_gb = total_vram_used / (1024 ** 3)
available_vram_gb = max(0.0, OLLAMA_VRAM_BUDGET_GB - total_vram_used_gb)

print(f"📊 Ollama VRAM Budget:    {OLLAMA_VRAM_BUDGET_GB:.2f} GB (GPU 1)")
print(f"🔥 Active VRAM Used:      {total_vram_used_gb:.2f} GB ({len(models)} model(s) loaded)")
print(f"🟢 Available VRAM:         {available_vram_gb:.2f} GB\n")

active_rows = []
for m in models:
    size_vram = (m.get("size_vram") if isinstance(m, dict) else getattr(m, "size_vram", 0)) or 0
    size_vram_gb = size_vram / (1024 ** 3)
    name = (m.get("name") or m.get("model")) if isinstance(m, dict) else (getattr(m, "name", None) or getattr(m, "model", "N/A"))
    processor = (m.get("processor") if isinstance(m, dict) else getattr(m, "processor", "100% GPU")) or "100% GPU"
    expires = (m.get("expires_at") if isinstance(m, dict) else getattr(m, "expires_at", "N/A"))
    
    active_rows.append({
        "Model": name,
        "VRAM Used": f"{size_vram_gb:.2f} GB",
        "Processor": processor,
        "Expires At": str(expires)[:19] if expires else "N/A",
    })

if active_rows:
    display(pd.DataFrame(active_rows))
else:
    print(f"ℹ️ No models currently loaded in GPU VRAM (Ollama is idle). Full {OLLAMA_VRAM_BUDGET_GB:.2f} GB available.")

📊 Ollama VRAM Budget:    44.00 GB (GPU 1)
🔥 Active VRAM Used:      17.00 GB (1 model(s) loaded)
🟢 Available VRAM:         27.00 GB



,Model,VRAM Used,Processor,Expires At
0,qwen3.8:27b,17.00 GB,100% GPU,2026-11-16 03:37:11


## 3. Download / Pull a Model
Download a model directly from the Ollama library (`client.pull()`).

*Note: The server pulls models through the institutional CSIM forward proxy (`http://192.41.170.82:3128`).*

In [4]:
# Target model to download
# target_model = "llama3.2:1b"
target_model = "qwen3.8:27b"
print(f"Pulling '{target_model}'...\n")

for chunk in client.pull(target_model, stream=True):
    status = chunk.get("status", "") if isinstance(chunk, dict) else getattr(chunk, "status", "")
    total = (chunk.get("total") if isinstance(chunk, dict) else getattr(chunk, "total", 0)) or 0
    completed = (chunk.get("completed") if isinstance(chunk, dict) else getattr(chunk, "completed", 0)) or 0
    
    if total > 0:
        pct = (completed / total) * 100
        print(f"\r{status}: {pct:5.1f}% ({completed / 1024**2:.1f} MB / {total / 1024**2:.1f} MB)", end="", flush=True)
    else:
        print(f"\r{status}...", end="", flush=True)

print("\n\n✅ Model ready!")

Pulling 'qwen3.8:27b'...

success...nused layers....0% (0.0 MB / 0.0 MB)31.9 MB)

✅ Model ready!


## 4. Select / Switch Models (and Explicitly Unload)
Our Ollama instance is configured with `OLLAMA_MAX_LOADED_MODELS=1` to enforce our 50% GPU slice (24 GB ceiling).
- Simply calling a new model automatically evicts the previous one from VRAM.
- You can also **explicitly unload** a model immediately by passing `keep_alive=0`.

In [5]:
def unload_model(model_name: str):
    """Instantly frees GPU VRAM by evicting the model from memory."""
    client.generate(model=model_name, prompt="", keep_alive=0)
    print(f"🗑️ Evicted '{model_name}' from GPU VRAM.")

# Check active models and unload if necessary
current_active = client.ps().get("models", [])
for m in current_active:
    name = m.get("name") or m.get("model")
    unload_model(name)

# Select the active model for our tests
# SELECTED_MODEL = "llama3.2:1b"
SELECTED_MODEL = "qwen3.8:27b"
print(f"🎯 Selected Model: {SELECTED_MODEL}")

🗑️ Evicted 'qwen3.8:27b' from GPU VRAM.
🎯 Selected Model: qwen3.8:27b


## 5. Use the Model (Chat Completion with Streaming)
Send prompt messages to the model and stream token output in real-time.

In [6]:
messages = [
    {"role": "system", "content": "You are a concise AI research assistant in AIT Brainlab."},
    {"role": "user", "content": "Explain the difference between full fine-tuning and LoRA in 3 bullet points."}
]

print(f"🤖 Streaming response from {SELECTED_MODEL}:\n")
stream = client.chat(
    model=SELECTED_MODEL,
    messages=messages,
    options={"num_ctx": 4096},  # Bound context window to prevent VRAM allocation spikes
    stream=True
)

for chunk in stream:
    print(chunk["message"]["content"], end="", flush=True)
print("\n")

🤖 Streaming response from qwen3.8:27b:

- **What gets updated:** Full fine-tuning adjusts *every* weight in the model, while LoRA freezes the base weights and injects small low-rank matrices (rank *r* ≪ hidden dim) only into a few layers (typically attention Q/V), so you train a few hundred thousand parameters instead of billions.

- **Memory & compute cost:** Full FT needs optimizer states and gradients for 100 % of parameters, making 7 B+ models impractical on a single GPU; LoRA reduces trainable parameters to ~0.1–1 % of the original, cutting GPU memory by roughly 3–10× and enabling multi-adapter training on consumer hardware.

- **Practical trade-off:** Full FT can overfit on small datasets and cause catastrophic forgetting of pre-trained knowledge; LoRA's inductive bias (low-rank constraint) acts as a regularizer, yields competitive quality on narrow tasks, and lets you swap, mix, or hot-reload adapters without reloading the entire base model.



## 6. Change Parameters & Keep-Alive (Make Model Active Longer)
You can customize inference parameters and control memory residency per-request:
- **`keep_alive`**: Extends how long the model stays loaded in GPU VRAM (e.g. `"1h"`, `"30m"`, or `-1` for indefinitely).
- **`options`**: Set `temperature`, `num_ctx` (context window size), `top_p`, etc.

In [7]:
# Call the model with custom parameters and a 1-hour VRAM keep-alive
response = client.chat(
    model=SELECTED_MODEL,
    messages=[
        {"role": "user", "content": "Write a haiku about GPU clusters in an academic lab."}
    ],
    options={
        "temperature": 0.3,    # Lower for more focused, deterministic output
        "top_p": 0.9,
        "num_ctx": 4096,       # Allocate 4096 tokens context window
    },
    keep_alive="1h"            # Keep model loaded in GPU VRAM for 1 hour!
)

print("Response:")
print(response["message"]["content"])

# Verify that the expiration time in VRAM is now extended
print("\n--- Active VRAM Status ---")
for m in client.ps().get("models", []):
    vram_gb = m.get("size_vram", 0) / (1024 ** 3)
    print(f"Model: {m.get('name')} | VRAM: {vram_gb:.2f} GB | Expires At: {m.get('expires_at')}")

Response:
Fans hum through the night,
green LEDs blink in the dark—
models train at last.

--- Active VRAM Status ---
Model: qwen3.8:27b | VRAM: 17.00 GB | Expires At: 2026-11-16 03:38:18.229678+00:00
